# ACTIVA M1 — Step 4 gate v2: frame pipeline variants

**Settings:** GPU T4, Internet ON. Same 4 manifest videos as v1 (P02_13, P03_21, P26_30, P28_23).

v1 (PyAV decode + PIL bicubic, BGR, whole frame) reached feature cosine 0.937 against the published
ActionFormer features. v2 tries to reproduce EPIC's own frame pipeline with ffmpeg
(resample to 60 fps, scale to 456x256, optional JPEG round trip like the `rgb_frames` dumps,
keep every 2nd frame) and reports decode/model time. Windows use the `floor` rule (T = n // 16).
No annotations are used. Output: `/kaggle/working/m1_step4_gate_v2.json`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!ffmpeg -version | head -1
import sys, torch, numpy; print(sys.version.split()[0], "torch", torch.__version__, "numpy", numpy.__version__)

In [ ]:
%cd /kaggle/working
!rm -rf ACTIVA dl
!git clone -q -b m1-real-action-inference https://github.com/harshjha15335/ACTIVA.git && git -C ACTIVA log --oneline -1
!git clone -q https://github.com/epic-kitchens/epic-kitchens-download-scripts.git dl
!pip install -q fvcore iopath simplejson av gdown pytest
import sys; sys.path[:0] = ["/kaggle/working/ACTIVA/code"]

## Unit tests (ffmpeg is available here, so the ffmpeg-path tests run too)

In [ ]:
!cd /kaggle/working/ACTIVA && python -m pytest -q tests/test_slowfast_features.py tests/test_actions.py tests/test_nms_1d_py.py 2>&1 | tail -3

## Assets (pinned, hashed) and raw videos

In [ ]:
!cd /kaggle/working/ACTIVA && python scripts/setup_m1_assets.py --dest /tmp/m1 > /kaggle/working/setup_m1.log 2>&1; grep -A1 '"SlowFast.pyth"' -m1 /kaggle/working/setup_m1.log
VIDEOS = ["P02_13", "P03_21", "P26_30", "P28_23"]
!cd /kaggle/working/dl && python epic_downloader.py --videos --specific-videos {",".join(VIDEOS)} --output-path /tmp/epic > /kaggle/working/dl.log 2>&1; tail -1 /kaggle/working/dl.log
import glob
MP4 = {v: glob.glob(f"/tmp/epic/**/{v}.MP4", recursive=True)[0] for v in VIDEOS}
print(MP4)

## Published features (system tar, only the 4 files)

In [ ]:
import os, subprocess, numpy as np
TAR = "/tmp/epic_kitchens.tar.gz"
if not os.path.exists(TAR):
    subprocess.run(["gdown", "--fuzzy", "https://drive.google.com/file/d/1Z4U_dLuu6_cV5NBIrSzsSDOOj2Uar85X/view", "-O", TAR], check=True)
os.makedirs("/tmp/pub", exist_ok=True)
subprocess.run(["tar", "-xzf", TAR, "-C", "/tmp/pub", "--wildcards"] + [f"*/{v}.npz" for v in VIDEOS], check=True)
PUB = {}
for v in VIDEOS:
    f = np.load(glob.glob(f"/tmp/pub/**/{v}.npz", recursive=True)[0]); PUB[v] = {k: f[k] for k in f.files}
    print(v, {k: x.shape for k, x in PUB[v].items()})

## Variants

In [ ]:
import time, json, pathlib
import slowfast_features as sf, actions, video_io
from slowfast_features import Preprocess, FrameSource
from action_config import EngineAssets
A = EngineAssets(pathlib.Path("/tmp/m1"))
bundle = sf.load_slowfast(A.slowfast_code, A.slowfast_cfg, A.slowfast_ckpt, device="cuda")
P = Preprocess(color="bgr")
VARIANTS = {
    "pyav_pil":      FrameSource(backend="pyav"),
    "ff_raw_even":   FrameSource(),
    "ff_raw_odd":    FrameSource(offset=1),
    "ff_jpg4_even":  FrameSource(jpeg_q=4),
    "ff_jpg4_odd":   FrameSource(jpeg_q=4, offset=1),
    "ff_jpg2_even":  FrameSource(jpeg_q=2),
    "ff30_raw":      FrameSource(fps=30.0, step=1),
}

def cos_rows(a, b):
    a = a / (np.linalg.norm(a, axis=1, keepdims=True) + 1e-12); b = b / (np.linalg.norm(b, axis=1, keepdims=True) + 1e-12)
    return (a * b).sum(1)

REPORT, FEATS = {}, {}
for name, src in VARIANTS.items():
    for v in VIDEOS:
        t0 = time.time()
        frames = list(sf.ffmpeg_frames(MP4[v], P, src) if src.backend == "ffmpeg"
                      else (sf.resize_frame(rgb, Preprocess(color="rgb")) for _t, rgb in video_io.iter_frames(MP4[v], 30.0)))
        t_dec = time.time() - t0
        t1 = time.time()
        out = sf.extract_features(None, bundle, P if src.backend == "ffmpeg" else Preprocess(color="bgr", width=456),
                                  batch_size=8, frames=iter(frames))
        torch.cuda.synchronize(); t_mod = time.time() - t1
        pub = PUB[v]; T = min(len(out["feats"]), len(pub["feats"]))
        c = cos_rows(out["feats"][:T], pub["feats"][:T])
        rec = {"n_frames": len(frames), "T_ours": int(len(out["feats"])), "T_pub": int(len(pub["feats"])),
               "feat_cos_mean": float(c.mean()), "feat_cos_min": float(c.min()),
               "verb_top1_agree": float((out["v_prob"][:T].argmax(1) == pub["v_prob"][:T].argmax(1)).mean()),
               "noun_top1_agree": float((out["n_prob"][:T].argmax(1) == pub["n_prob"][:T].argmax(1)).mean()),
               "decode_s": round(t_dec, 2), "model_s": round(t_mod, 2)}
        REPORT.setdefault(name, {})[v] = rec; FEATS[(name, v)] = out["feats"]
        print(f"{name:13s} {v}: T {rec['T_ours']}/{rec['T_pub']} cos {rec['feat_cos_mean']:.4f} (min {rec['feat_cos_min']:.3f}) "
              f"verb {rec['verb_top1_agree']:.2f} noun {rec['noun_top1_agree']:.2f} decode {t_dec:.1f}s model {t_mod:.1f}s")
SUMMARY = {n: {k: float(np.mean([r[k] for r in vids.values()])) for k in
               ("feat_cos_mean", "feat_cos_min", "verb_top1_agree", "noun_top1_agree", "decode_s", "model_s")}
           for n, vids in REPORT.items()}
for n, s in sorted(SUMMARY.items(), key=lambda kv: -kv[1]["feat_cos_mean"]):
    print(f"{n:13s}", {k: round(x, 4) for k, x in s.items()})

## Downstream for the best variant: ActionFormer detections ours vs published

In [ ]:
best = max(SUMMARY, key=lambda n: SUMMARY[n]["feat_cos_mean"]); print("best:", best)
RUN = {k: actions.ActionFormerRunner(A.af_code, A.af_cfg(k), actions._latest_ckpt(A.af_ckpt_dir(k)), "cuda") for k in ("verb", "noun")}

def match_rate(da, db, k=100, tiou=0.5):
    ia, ib = np.argsort(-da[2])[:k], np.argsort(-db[2])[:k]
    m = actions.tiou_matrix(da[0][ia], db[0][ib]) >= tiou
    return float((m & (da[1][ia][:, None] == db[1][ib][None, :])).any(1).mean())

DOWN = {}
for v in VIDEOS:
    dur = video_io.inspect_video(MP4[v]).duration
    DOWN[v] = {k: {"top100_ours_in_pub": match_rate(RUN[k].detect(FEATS[(best, v)], dur), RUN[k].detect(PUB[v]["feats"], dur))}
               for k in ("verb", "noun")}
    print(v, DOWN[v])
json.dump({"commit": subprocess.run(["git", "-C", "/kaggle/working/ACTIVA", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip(),
           "torch": torch.__version__, "gpu": torch.cuda.get_device_name(0),
           "summary": SUMMARY, "per_video": REPORT, "best": best, "downstream": DOWN},
          open("/kaggle/working/m1_step4_gate_v2.json", "w"), indent=1)
print("saved /kaggle/working/m1_step4_gate_v2.json")